# Custom CNN Baseline - Seed Germination

This notebook trains, validates, and tests the Custom CNN baseline for binary seed crop classification: `germinated` vs `non_germinated`.

Expected dataset layout:

```text
/content/data/crops/train/germinated/
/content/data/crops/train/non_germinated/
/content/data/crops/val/germinated/
/content/data/crops/val/non_germinated/
/content/data/crops/test/germinated/
/content/data/crops/test/non_germinated/
```


## Cell 1 - Kiểm tra môi trường Colab

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Cell 2 - Import thư viện cần dùng

In [ ]:
import json
import random
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


## Cell 3 - Cấu hình chung cho model

In [ ]:
IMAGE_SIZE = 224
INPUT_CHANNELS = 3
NUM_CLASSES = 2
DROPOUT = 0.30

# torchvision.datasets.ImageFolder sorts class folders alphabetically.
# With the current folder names: germinated -> 0, non_germinated -> 1.
CLASS_NAMES = ["germinated", "non_germinated"]
CLASS_TO_IDX = {name: index for index, name in enumerate(CLASS_NAMES)}

print("Default class mapping:", CLASS_TO_IDX)


## Cell 4 - Transform ảnh crop đầu vào

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.02,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

print(train_transform)
print(eval_transform)

## Cell 5 - Kiểm tra dataset ImageFolder nếu đã upload crop data

In [ ]:
# Change these paths if your crops dataset is uploaded elsewhere.
CROP_ROOT = Path("/content/data/crops")
CROPS_ZIP = Path("/content/drive/MyDrive/crops.zip")


def maybe_extract_crops_zip(zip_path: Path, crop_root: Path) -> None:
    """Extract crops.zip if crop_root does not exist.

    Supported zip layouts:
    - data/crops/train/...
    - crops/train/...
    - train/...
    """

    if crop_root.exists():
        return
    if not zip_path.exists():
        print(f"Crop dataset not found at: {crop_root}")
        print(f"Zip file not found at: {zip_path}")
        return

    print(f"Extracting: {zip_path}")
    with zipfile.ZipFile(zip_path) as zip_file:
        names = zip_file.namelist()
        if any(name.startswith("data/crops/") for name in names):
            extract_dir = Path("/content")
        elif any(name.startswith("crops/") for name in names):
            extract_dir = Path("/content/data")
        else:
            extract_dir = crop_root
        extract_dir.mkdir(parents=True, exist_ok=True)
        zip_file.extractall(extract_dir)
    print(f"Extracted to: {extract_dir}")


maybe_extract_crops_zip(CROPS_ZIP, CROP_ROOT)

required_dirs = [
    CROP_ROOT / "train" / "germinated",
    CROP_ROOT / "train" / "non_germinated",
    CROP_ROOT / "val" / "germinated",
    CROP_ROOT / "val" / "non_germinated",
    CROP_ROOT / "test" / "germinated",
    CROP_ROOT / "test" / "non_germinated",
]
missing_dirs = [path for path in required_dirs if not path.exists()]
if missing_dirs:
    raise FileNotFoundError(
        "Crop dataset layout is invalid. Missing:\n"
        + "\n".join(str(path) for path in missing_dirs)
    )

train_dataset = datasets.ImageFolder(CROP_ROOT / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(CROP_ROOT / "val", transform=eval_transform)
test_dataset = datasets.ImageFolder(CROP_ROOT / "test", transform=eval_transform)

CLASS_NAMES = train_dataset.classes
CLASS_TO_IDX = train_dataset.class_to_idx

print("Crop root:", CROP_ROOT)
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("ImageFolder class_to_idx:", CLASS_TO_IDX)
print("Class names:", CLASS_NAMES)


## Cell 6 - Định nghĩa một block convolution

In [ ]:
class ConvBlock(nn.Module):
    """Convolution + BatchNorm + ReLU + MaxPool."""

    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

## Cell 7 - Định nghĩa Custom CNN

In [ ]:
class CustomCNN(nn.Module):
    """Custom CNN baseline cho phân loại crop hạt giống 2 lớp."""

    def __init__(
        self,
        input_channels: int = 3,
        num_classes: int = 2,
        dropout: float = 0.30,
    ) -> None:
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(input_channels, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x)
        logits = self.classifier(x)
        return logits

## Cell 8 - Hàm tạo model và đếm tham số

In [ ]:
def build_custom_cnn(
    input_channels: int = INPUT_CHANNELS,
    num_classes: int = NUM_CLASSES,
    dropout: float = DROPOUT,
) -> CustomCNN:
    return CustomCNN(
        input_channels=input_channels,
        num_classes=num_classes,
        dropout=dropout,
    )


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

## Cell 9 - Khởi tạo model

In [ ]:
model = build_custom_cnn().to(device)

print(model)
print("Trainable parameters:", f"{count_trainable_parameters(model):,}")

## Cell 10 - Kiểm tra forward pass

In [ ]:
batch_size = 4
sample_input = torch.randn(batch_size, INPUT_CHANNELS, IMAGE_SIZE, IMAGE_SIZE).to(device)

model.eval()
with torch.no_grad():
    logits = model(sample_input)
    probabilities = torch.softmax(logits, dim=1)
    predicted_class_ids = torch.argmax(probabilities, dim=1)

print("Input shape:", tuple(sample_input.shape))
print("Logits shape:", tuple(logits.shape))
print("Probabilities shape:", tuple(probabilities.shape))
print("Predicted class ids:", predicted_class_ids.cpu().tolist())
print("Predicted labels:", [CLASS_NAMES[i] for i in predicted_class_ids.cpu().tolist()])

## Cell 11 - Kiểm tra shape qua từng phần của model

In [ ]:
x = torch.randn(1, INPUT_CHANNELS, IMAGE_SIZE, IMAGE_SIZE).to(device)

model.eval()
with torch.no_grad():
    features = model.features(x)
    pooled = model.pool(features)
    output = model.classifier(pooled)

print("Input:", tuple(x.shape))
print("After feature extractor:", tuple(features.shape))
print("After adaptive pooling:", tuple(pooled.shape))
print("Final logits:", tuple(output.shape))

## Cell 12 - Mô tả ngắn kiến trúc để đưa vào báo cáo

In [ ]:
architecture_summary = f"""
Custom CNN baseline nhận ảnh crop RGB kích thước {IMAGE_SIZE} x {IMAGE_SIZE}.
Feature extractor gồm 3 block Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d,
với số kênh lần lượt là 32, 64 và 128.
Sau feature extractor, model dùng AdaptiveAvgPool2d để đưa feature map về 1 x 1,
sau đó dùng classifier gồm Dropout, Linear(128, 64), ReLU, Dropout và Linear(64, 2).
Model trả về logits 2 lớp và được huấn luyện bằng CrossEntropyLoss.
Tổng số tham số trainable: {count_trainable_parameters(model):,}.
""".strip()

print(architecture_summary)

## Cell 13 - Training configuration

By default, this notebook trains on the full crop dataset. If Colab runs out of RAM/time, set `USE_SUBSET = True` for a smaller baseline run.


In [ ]:
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

USE_SUBSET = False
MAX_TRAIN_SAMPLES = 30000
MAX_VAL_SAMPLES = 6000
MAX_TEST_SAMPLES = 6000

OUTPUT_DIR = Path("/content/outputs/baseline_cnn")
CHECKPOINT_PATH = OUTPUT_DIR / "best_custom_cnn.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
METRICS_PATH = OUTPUT_DIR / "test_metrics.csv"
SUMMARY_PATH = OUTPUT_DIR / "test_summary.json"
CONFUSION_MATRIX_PATH = OUTPUT_DIR / "confusion_matrix.png"
LEARNING_CURVES_PATH = OUTPUT_DIR / "learning_curves.png"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output dir:", OUTPUT_DIR)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Use subset:", USE_SUBSET)


## Cell 14 - Optional subset and DataLoader


In [ ]:
def make_stratified_subset(dataset, max_samples: int, seed: int = SEED):
    if max_samples is None or max_samples <= 0 or max_samples >= len(dataset):
        return dataset

    targets = np.array(dataset.targets)
    rng = np.random.default_rng(seed)
    selected_indices = []
    per_class = max_samples // len(dataset.classes)

    for class_id in range(len(dataset.classes)):
        class_indices = np.where(targets == class_id)[0]
        rng.shuffle(class_indices)
        selected_indices.extend(class_indices[: min(per_class, len(class_indices))].tolist())

    rng.shuffle(selected_indices)
    return Subset(dataset, selected_indices)


train_data = make_stratified_subset(train_dataset, MAX_TRAIN_SAMPLES) if USE_SUBSET else train_dataset
val_data = make_stratified_subset(val_dataset, MAX_VAL_SAMPLES) if USE_SUBSET else val_dataset
test_data = make_stratified_subset(test_dataset, MAX_TEST_SAMPLES) if USE_SUBSET else test_dataset

pin_memory = torch.cuda.is_available()
loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": pin_memory,
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_data, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_data, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_data, shuffle=False, **loader_kwargs)

print("Train batches:", len(train_loader), "samples:", len(train_data))
print("Val batches:", len(val_loader), "samples:", len(val_data))
print("Test batches:", len(test_loader), "samples:", len(test_data))


## Cell 15 - Metrics and epoch runner


In [ ]:
def confusion_matrix_np(y_true, y_pred, num_classes: int) -> np.ndarray:
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for target, pred in zip(y_true, y_pred):
        matrix[int(target), int(pred)] += 1
    return matrix


def metrics_from_confusion_matrix(matrix: np.ndarray, class_names: list[str]) -> tuple[dict, pd.DataFrame]:
    total = matrix.sum()
    accuracy = float(np.trace(matrix) / total) if total else 0.0

    rows = []
    precisions = []
    recalls = []
    f1_scores = []
    supports = []

    for class_id, class_name in enumerate(class_names):
        tp = matrix[class_id, class_id]
        fp = matrix[:, class_id].sum() - tp
        fn = matrix[class_id, :].sum() - tp
        support = matrix[class_id, :].sum()

        precision = float(tp / (tp + fp)) if (tp + fp) else 0.0
        recall = float(tp / (tp + fn)) if (tp + fn) else 0.0
        f1 = float(2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

        rows.append(
            {
                "class": class_name,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "support": int(support),
            }
        )
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        supports.append(support)

    macro = {
        "accuracy": accuracy,
        "macro_precision": float(np.mean(precisions)),
        "macro_recall": float(np.mean(recalls)),
        "macro_f1": float(np.mean(f1_scores)),
    }

    supports_arr = np.array(supports, dtype=np.float64)
    if supports_arr.sum() > 0:
        macro["weighted_precision"] = float(np.average(precisions, weights=supports_arr))
        macro["weighted_recall"] = float(np.average(recalls, weights=supports_arr))
        macro["weighted_f1"] = float(np.average(f1_scores, weights=supports_arr))
    else:
        macro["weighted_precision"] = 0.0
        macro["weighted_recall"] = 0.0
        macro["weighted_f1"] = 0.0

    return macro, pd.DataFrame(rows)


def evaluate_predictions(y_true, y_pred, class_names: list[str]) -> tuple[dict, pd.DataFrame, np.ndarray]:
    matrix = confusion_matrix_np(y_true, y_pred, len(class_names))
    summary, per_class = metrics_from_confusion_matrix(matrix, class_names)
    return summary, per_class, matrix


def run_one_epoch(model, dataloader, criterion, optimizer=None, desc: str = ""):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_targets = []
    all_preds = []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        progress = tqdm(dataloader, desc=desc, leave=False)
        for images, targets in progress:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = criterion(logits, targets)

            if is_train:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            preds = torch.argmax(logits, dim=1)
            all_targets.extend(targets.detach().cpu().tolist())
            all_preds.extend(preds.detach().cpu().tolist())

            running_loss = total_loss / max(1, len(all_targets))
            progress.set_postfix(loss=f"{running_loss:.4f}")

    avg_loss = total_loss / max(1, len(all_targets))
    summary, per_class, matrix = evaluate_predictions(all_targets, all_preds, CLASS_NAMES)
    summary["loss"] = avg_loss
    return summary, per_class, matrix, all_targets, all_preds


## Cell 16 - Train Custom CNN baseline


In [ ]:
model = build_custom_cnn().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

history = []
best_val_f1 = -1.0
best_epoch = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_summary, _, _, _, _ = run_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        desc=f"Epoch {epoch}/{EPOCHS} train",
    )
    val_summary, _, _, _, _ = run_one_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
        desc=f"Epoch {epoch}/{EPOCHS} val",
    )

    row = {
        "epoch": epoch,
        "train_loss": train_summary["loss"],
        "train_accuracy": train_summary["accuracy"],
        "train_macro_f1": train_summary["macro_f1"],
        "val_loss": val_summary["loss"],
        "val_accuracy": val_summary["accuracy"],
        "val_macro_f1": val_summary["macro_f1"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={row['train_loss']:.4f} train_f1={row['train_macro_f1']:.4f} | "
        f"val_loss={row['val_loss']:.4f} val_f1={row['val_macro_f1']:.4f}"
    )

    if val_summary["macro_f1"] > best_val_f1:
        best_val_f1 = val_summary["macro_f1"]
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "class_names": CLASS_NAMES,
                "class_to_idx": CLASS_TO_IDX,
                "config": {
                    "image_size": IMAGE_SIZE,
                    "batch_size": BATCH_SIZE,
                    "epochs": EPOCHS,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "dropout": DROPOUT,
                    "use_subset": USE_SUBSET,
                    "max_train_samples": MAX_TRAIN_SAMPLES if USE_SUBSET else None,
                    "max_val_samples": MAX_VAL_SAMPLES if USE_SUBSET else None,
                    "max_test_samples": MAX_TEST_SAMPLES if USE_SUBSET else None,
                },
                "val_summary": val_summary,
            },
            CHECKPOINT_PATH,
        )
        print(f"  Saved best checkpoint: {CHECKPOINT_PATH}")

elapsed_minutes = (time.time() - start_time) / 60
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print(f"Best epoch: {best_epoch} | best val macro F1: {best_val_f1:.4f}")
print(f"Training time: {elapsed_minutes:.2f} minutes")
print(f"Saved history: {HISTORY_PATH}")


## Cell 17 - Learning curves


In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"], history_df["train_loss"], label="train")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="train")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Macro F1")
plt.legend()

plt.tight_layout()
plt.savefig(LEARNING_CURVES_PATH, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved learning curves: {LEARNING_CURVES_PATH}")


## Cell 18 - Test evaluation with the best checkpoint


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model = build_custom_cnn().to(device)
model.load_state_dict(checkpoint["model_state_dict"])

print("Loaded checkpoint:", CHECKPOINT_PATH)
print("Best epoch:", checkpoint["epoch"])
print("Class names:", checkpoint["class_names"])

test_summary, test_per_class, test_matrix, test_targets, test_preds = run_one_epoch(
    model,
    test_loader,
    criterion,
    optimizer=None,
    desc="Test",
)

test_per_class.to_csv(METRICS_PATH, index=False)
SUMMARY_PATH.write_text(
    json.dumps(test_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Test summary:")
for key, value in test_summary.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

print("\nPer-class metrics:")
display(test_per_class)
print(f"Saved per-class metrics: {METRICS_PATH}")
print(f"Saved test summary: {SUMMARY_PATH}")


## Cell 19 - Confusion matrix


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    test_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Custom CNN Test Confusion Matrix")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved confusion matrix: {CONFUSION_MATRIX_PATH}")


## Cell 20 - Demo predictions on test images


In [ ]:
def denormalize_image(tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = tensor.cpu() * std + mean
    image = image.clamp(0, 1)
    return image.permute(1, 2, 0).numpy()


model.eval()
num_demo = min(12, len(test_dataset))
rng = np.random.default_rng(SEED)
demo_indices = rng.choice(len(test_dataset), size=num_demo, replace=False)

plt.figure(figsize=(12, 8))
with torch.no_grad():
    for plot_id, dataset_idx in enumerate(demo_indices, start=1):
        image, true_label = test_dataset[int(dataset_idx)]
        logits = model(image.unsqueeze(0).to(device))
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu()
        pred_label = int(torch.argmax(probs).item())
        confidence = float(probs[pred_label].item())

        plt.subplot(3, 4, plot_id)
        plt.imshow(denormalize_image(image))
        plt.axis("off")
        color = "green" if pred_label == true_label else "red"
        plt.title(
            f"GT: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[pred_label]} ({confidence:.2%})",
            color=color,
            fontsize=9,
        )

plt.tight_layout()
plt.show()


## Cell 21 - Zip result files for download


In [ ]:
import shutil

archive_base = Path("/content/baseline_cnn_results")
archive_path = shutil.make_archive(str(archive_base), "zip", OUTPUT_DIR)
print("Result archive:", archive_path)

# If running in Colab and you want to download the result archive, uncomment:
# from google.colab import files
# files.download(archive_path)
